### Model Validation Report — Probability of Default (Logistic Regression), FIM500 2-Lines-of-Defense Project

| Field | Entry |
|---|---|
| Model name | Probability of Default model (scikit-learn `LogisticRegression`), "Give Me Some Credit" data |
| Model developer (1LOD) | Anna Pope |
| Model validator (2LOD) | Nikhil Srivastava |
| Model version / GitHub commit reviewed | Version 1 |
| Model tier (1 / 2 / 3) | Model Tier 1|
| Date of validation | 2026-11-9 |
| Overall conclusion | **Approved with conditions** |

#### *Executive Summary*

The model scores each existing borrower's probability of serious delinquency (90+ days past due within two years) with a logistic regression on ten credit-bureau-style variables. **Conclusion: Approved with conditions.** The logistic regression approach is appropriate and the project is well organised. Three items must be fixed before the model is used: `model.py` needs two missing saved values to run, a capping step is applied when scoring but not when training, and the headline recall/precision/accuracy figures were calculated on training data rather than the hold-out. The outcome-based tests (replication, AUC/KS/calibration, challenger) are still to be completed because the training file `cs-training.csv` and the saved model artifacts were not available to the validator, and `cs-test.csv` has no target labels.

| Severity | Number of findings |
|---|---|
| High | 3 (F1–F3) |
| Medium | 6 (F4–F9) |
| Low | 2 (F10–F11) |

#### *i. Scope of Review*

**Reviewed:** model documentation notebook; `model.ipynb` (code and stored outputs, read in full); `model.py` (executed); `cs-test.csv` (101,503 rows, profiled and scored).

**Executed:** the original `model.py` and a corrected copy, both run against `cs-test.csv`. Because the pickles were not supplied, they were run with **placeholder artifacts** (random labels). That proves whether the *code path* works, and says nothing about model quality.

**Deliberately not tested, and why:**
- *Replication of AUC 0.8088 / recall 34.68% / precision 31.32% / accuracy 90.55%* — needs `cs-training.csv` (not supplied; the sandbox could not download it).
- *Gini, KS, decile calibration, cross-validation, threshold sensitivity, VIF on training data, log-odds linearity (Box-Tidwell), challenger model* — need labelled data. `cs-test.csv` is Kaggle's hold-out and every `SeriousDlqin2yrs` value is empty.
- *`model.ipynb` was not re-executed*: it reads the developer's absolute laptop path.

These items are listed as conditions for re-submission in section xi; sections vi and vii are therefore **incomplete by necessity**, not passed.

#### *ii. Purpose, Business Context and Intended Use*
*Mirrors developer sections i–iii*

- **Purpose:** clear — estimate PD for existing borrowers so that action can be taken when PD is high.
- **Intended use / scope:** stated (current, already-approved borrowers; exact column names). **Not stated:** what the PD is used for (pricing, limits, provisioning, collections), the decision threshold's business cost, or what the model must *not* be used for (e.g. new-applicant underwriting, IFRS 9 / CECL provisioning, regulatory capital). The output is a two-year serious-delinquency probability, which is not the same as a default definition used for provisioning.
- **Out-of-scope use:** the documentation states `cs-test.csv` is "the dataset that should be used" for validation. It cannot be, because it has no labels (F3).
- **Population fit (F9):** the data is a 2011 public Kaggle competition set whose source and time frame the developer says are unknown, and which describes credit-bureau snapshots of borrowers, while the model is positioned for a bank's *existing* book. No evidence is provided that the population is comparable.

#### *iii. Data Review*
*Mirrors developer data section*

- **Source / period:** Kaggle "Give Me Some Credit" (2011); period unknown (stated by developer). Training file: ~150k rows (public description; **not verified here**). Test file: 101,503 rows.
- **Target:** `SeriousDlqin2yrs` = 90+ days past due within 2 years; definition is clear. The documentation says "only 6%" defaulted; the public description gives ~6.7% (**not verified here**). Imbalance is not addressed (F7).
- **Cleaning:** median imputation of `MonthlyIncome` and `NumberOfDependents` (medians from `x_train` only — good); documentation mentions only income. Capping of utilization (85th pct) and income (75th pct) is **undocumented and not applied in training** (F2, F5).
- **Split:** stratified 80/20 with `random_state=42` — appropriate. No time-based split is possible (no dates).
- **Findings on `cs-test.csv`** (run below): 20,103 missing incomes (19.8%), 2,626 missing dependents; **214 rows with sentinel values 96/98** in all three delinquency counts; 153 rows with utilization above 10 (max 21,821); 19,397 rows with `DebtRatio` above 10, and 18,849 of the 20,103 missing-income rows have `DebtRatio` > 1 (the field holds a *dollar amount* when income is missing); 1,020 zero incomes; 328 duplicate rows; ages 21–104.

**Leakage check:**
- *Availability at decision time:* all ten features are described as "previous two years" or current balances, so they look available at scoring time, but without dates the timing relative to the target window cannot be confirmed. `NumberOfTimes90DaysLate` is the same type of event as the target, so a borrower already 90+ late is near-trivially predicted; this should be stated.
- *Preprocessing fitted on training data only:* **imputation medians and the scaler: yes** (good). **Capping bounds: no** — `upper_bound` is computed on the full `training_data` (including the hold-out rows), a mild leak, and then not used in fitting anyway (F2).

In [1]:
import pandas as pd, numpy as np
d = pd.read_csv("cs-test.csv", index_col=0)
print("rows, cols:", d.shape)
print("target non-null (cs-test labels):", d["SeriousDlqin2yrs"].notna().sum())
print("\nMissing values:\n", d.isna().sum()[d.isna().sum() > 0])
print("\nExact duplicate rows:", d.duplicated().sum())
dq = ["NumberOfTime30-59DaysPastDueNotWorse", "NumberOfTimes90DaysLate", "NumberOfTime60-89DaysPastDueNotWorse"]
print("\nSentinel codes (>=96) per delinquency column:", {c: int((d[c] >= 96).sum()) for c in dq})
print("Max excluding sentinels:", {c: int(d[c][d[c] < 96].max()) for c in dq})
print("\nUtilization >1:", int((d.RevolvingUtilizationOfUnsecuredLines > 1).sum()), "| >10:", int((d.RevolvingUtilizationOfUnsecuredLines > 10).sum()))
print("DebtRatio >1:", int((d.DebtRatio > 1).sum()), "| >10:", int((d.DebtRatio > 10).sum()))
print("DebtRatio>1 among missing-income rows:", int(((d.DebtRatio > 1) & d.MonthlyIncome.isna()).sum()), "of", int(d.MonthlyIncome.isna().sum()))
print("Zero income:", int((d.MonthlyIncome == 0).sum()), "| age range:", d.age.min(), "-", d.age.max())
print("\nValue ranges:\n", d.drop(columns="SeriousDlqin2yrs").describe().T[["min", "50%", "max"]].to_string())

rows, cols: (101503, 11)
target non-null (cs-test labels): 0

Missing values:
 SeriousDlqin2yrs      101503
MonthlyIncome          20103
NumberOfDependents      2626
dtype: int64

Exact duplicate rows: 328

Sentinel codes (>=96) per delinquency column: {'NumberOfTime30-59DaysPastDueNotWorse': 214, 'NumberOfTimes90DaysLate': 214, 'NumberOfTime60-89DaysPastDueNotWorse': 214}
Max excluding sentinels: {'NumberOfTime30-59DaysPastDueNotWorse': 19, 'NumberOfTimes90DaysLate': 18, 'NumberOfTime60-89DaysPastDueNotWorse': 9}

Utilization >1: 2181 | >10: 153
DebtRatio >1: 23578 | >10: 19397
DebtRatio>1 among missing-income rows: 18849 of 20103
Zero income: 1020 | age range: 21 - 104

Value ranges:
                                        min          50%        max
RevolvingUtilizationOfUnsecuredLines   0.0     0.152586    21821.0
age                                   21.0    52.000000      104.0
NumberOfTime30-59DaysPastDueNotWorse   0.0     0.000000       98.0
DebtRatio                           

#### *iv. Conceptual Soundness*
*Mirrors developer section iv*

Logistic regression is a sound, standard, explainable choice for PD, and a suitable baseline. Concerns:

- **Rejection of alternatives is not valid.** The documentation says other models were rejected because "none of them producing probabilities." Gradient boosting, random forests, naive Bayes and calibrated SVMs all produce probabilities. No alternative was built or compared (F7).
- **Feature justification** is reasonable qualitatively but no univariate/WoE analysis, sign check or significance test is shown; the age claim ("from data analysis performed before constructing the model") has no evidence in the notebook.
- **Imbalance and calibration:** unweighted fit on a rare event with no calibration check. The output is labelled "PD (%)", so calibration matters, but is never tested (F7).
- **Sentinel codes and DebtRatio:** see F6.

| Assumption | How tested | Result | Holds? |
|---|---|---|---|
| Binary outcome | By construction | `SeriousDlqin2yrs` is 0/1 | Yes |
| Independent observations | Not tested by developer. I looked for duplicates | No borrower ID; 328 exact duplicate rows in `cs-test.csv` | Unverified |
| No multicollinearity | Developer: none. Validator: correlation matrix on `cs-test.csv` (VIF on training data pending) | The three delinquency counts correlate at **0.99** with each other, driven by the 214 rows coded 96/98 (falls to 0.21–0.32 when those rows are excluded) | **Fails (indicative)** |
| Linearity of log-odds in the continuous predictors | Not tested by developer; needs labels | Predictors are heavy-tailed (utilization max 21,821; DebtRatio max 268,326), so linearity is doubtful | Not tested |
| Adequate events per variable | Not stated | ~10k events / 10 variables should be ample | Likely (unverified) |

In [2]:
x = d.drop(columns="SeriousDlqin2yrs")
print(x[dq].corr().round(3))
clean = x[(x[dq] < 96).all(axis=1)]
print("\nSame correlations excluding sentinel rows (n=%d):" % len(clean))
print(clean[dq].corr().round(3))

                                      NumberOfTime30-59DaysPastDueNotWorse  \
NumberOfTime30-59DaysPastDueNotWorse                                 1.000   
NumberOfTimes90DaysLate                                              0.986   
NumberOfTime60-89DaysPastDueNotWorse                                 0.989   

                                      NumberOfTimes90DaysLate  \
NumberOfTime30-59DaysPastDueNotWorse                    0.986   
NumberOfTimes90DaysLate                                 1.000   
NumberOfTime60-89DaysPastDueNotWorse                    0.994   

                                      NumberOfTime60-89DaysPastDueNotWorse  
NumberOfTime30-59DaysPastDueNotWorse                                 0.989  
NumberOfTimes90DaysLate                                              0.994  
NumberOfTime60-89DaysPastDueNotWorse                                 1.000  

Same correlations excluding sentinel rows (n=101289):
                                      NumberOfTime30-59DaysPast

#### *v. Replication*
*Mirrors developer section v*

Reproduction was attempted from the files supplied. Steps that could not be reproduced:
1. `model.ipynb` cell 0 reads `/Users/annakpope/Downloads/...` (not portable) and needs `cs-training.csv.zip` (not supplied).
2. `model.py` needs `my_model.pkl`, `scaling.pkl`, `median_income.pkl` (not supplied) and then fails with `NameError` even if they were present (F1; demonstrated below).
3. The documentation's usage example (`from model import predict_pd`) therefore fails as written.

| Reported metric | Developer's value | Replicated value | Match? |
|---|---|---|---|
| AUC (hold-out) | 0.8088 (0.81 in text) | Not replicated: no training data | Not tested |
| Recall @ cut-off | 34.68% | Not replicated | Not tested; computed in-sample (F3) |
| Precision @ cut-off | 31.32% | Not replicated | Not tested; computed in-sample (F3) |
| Accuracy @ cut-off | 90.55% | Not replicated | Not tested; computed in-sample (F3) |
| Cut-off | 25% (document) | 20% in code | **No** (F4) |

*Reading the stored output:* if prevalence is ~6.7% (public figure, not verified here), a model predicting "no default" for everyone would score ~93.3% accuracy, higher than the reported 90.55%, so the accuracy figure carries little information. All three figures are in-sample (F3).

In [3]:
# # Does the delivered model.py run on cs-test? Uses PLACEHOLDER artifacts (random labels) -- tests the code path only.
# import os, sys, tempfile, importlib.util, joblib
# from sklearn.preprocessing import StandardScaler
# from sklearn.linear_model import LogisticRegression
# F = ["MonthlyIncome","DebtRatio","NumberOfTimes90DaysLate","RevolvingUtilizationOfUnsecuredLines","NumberOfTime30-59DaysPastDueNotWorse",
#      "NumberOfTime60-89DaysPastDueNotWorse","NumberOfDependents","NumberOfOpenCreditLinesAndLoans","NumberRealEstateLoansOrLines","age"]
# xx = d[F].copy(); xx["MonthlyIncome"] = xx["MonthlyIncome"].fillna(xx["MonthlyIncome"].median()); xx["NumberOfDependents"] = xx["NumberOfDependents"].fillna(0)
# sc = StandardScaler().fit(xx); m = LogisticRegression().fit(sc.transform(xx), np.random.RandomState(0).randint(0, 2, len(xx)))
# here = os.getcwd(); tmp = tempfile.mkdtemp(); os.chdir(tmp)
# for n, o in [("my_model", m), ("scaling", sc), ("median_income", 5400.0), ("median_depend", 0.0), ("upper_bound", 0.856)]:
#     joblib.dump(o, n + ".pkl")
# def load(path, name):
#     s = importlib.util.spec_from_file_location(name, path); mod = importlib.util.module_from_spec(s); s.loader.exec_module(mod); return mod
# orig = load(os.path.join(here, "model.py"), "orig"); fixed = load(os.path.join(here, "model.py"), "fixed")
# try: orig.predict_pd(d)
# except Exception as e: print("ORIGINAL model.py ->", type(e).__name__ + ":", e)
# r = fixed.predict_pd(d)
# print("CORRECTED model.py -> rows:", len(r), "| NaN PDs:", int(r["PD (%)"].isna().sum()), "| all within 0-100:", bool(r["PD (%)"].between(0, 100).all()))
# os.chdir(here)

In [4]:
import os, tempfile, importlib.util, joblib
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Folder that holds model.py and original_model.py. Defaults to the folder this notebook runs in.
# If your files are elsewhere, replace with the full folder path (no filename), e.g. "/Users/you/Downloads/validation"
NOTEBOOK_DIR = os.getcwd()

orig_path  = os.path.join(NOTEBOOK_DIR, "original_model.py")   # developer's file, renamed from "model (1).py"
fixed_path = os.path.join(NOTEBOOK_DIR, "model.py")            # corrected file
for p in (orig_path, fixed_path):
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing {p}. Folder contents: {os.listdir(NOTEBOOK_DIR)}")

# Placeholder artifacts (random labels): tests that the code runs, NOT model quality
F = ["MonthlyIncome", "DebtRatio", "NumberOfTimes90DaysLate", "RevolvingUtilizationOfUnsecuredLines",
     "NumberOfTime30-59DaysPastDueNotWorse", "NumberOfTime60-89DaysPastDueNotWorse", "NumberOfDependents",
     "NumberOfOpenCreditLinesAndLoans", "NumberRealEstateLoansOrLines", "age"]
xx = d[F].copy()
xx["MonthlyIncome"] = xx["MonthlyIncome"].fillna(xx["MonthlyIncome"].median())
xx["NumberOfDependents"] = xx["NumberOfDependents"].fillna(0)
sc = StandardScaler().fit(xx)
m = LogisticRegression().fit(sc.transform(xx), np.random.RandomState(0).randint(0, 2, len(xx)))

def load(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

start = os.getcwd()
tmp = tempfile.mkdtemp()
try:
    os.chdir(tmp)   # both model files load their .pkl files from the current folder
    for n, o in [("my_model", m), ("scaling", sc), ("median_income", 5400.0),
                 ("median_depend", 0.0), ("upper_bound", 0.856)]:
        joblib.dump(o, n + ".pkl")

    orig  = load(orig_path, "orig")
    fixed = load(fixed_path, "fixed")

    try:
        orig.predict_pd(d)
        print("ORIGINAL model.py -> ran without error")
    except Exception as e:
        print("ORIGINAL model.py ->", type(e).__name__ + ":", e)

    r = fixed.predict_pd(d)
    print("CORRECTED model.py -> rows:", len(r),
          "| NaN PDs:", int(r["PD (%)"].isna().sum()),
          "| all within 0-100:", bool(r["PD (%)"].between(0, 100).all()))
finally:
    os.chdir(start)   # always return to the original folder

ORIGINAL model.py -> NameError: name 'median_depend' is not defined
CORRECTED model.py -> rows: 101503 | NaN PDs: 0 | all within 0-100: True


#### *vi. Independent Testing*

**Status: pending (no labelled out-of-sample data was available).** Planned tests, to run once `cs-training.csv` (and ideally the three pickles) are supplied:

| Test | Purpose | Status |
|---|---|---|
| AUC / Gini on a fresh hold-out | Discrimination (compare with 0.8088) | Pending |
| KS statistic | Separation | Pending |
| Calibration: mean predicted vs observed default rate by decile | Is "PD (%)" a probability? | Pending |
| 5-fold stratified CV and AUC by fold | Stability | Pending |
| Recall/precision across cut-offs 5–50% | Sensitivity to threshold (F4) | Pending |
| Effect of removing the 96/98 codes and of the 85th-pct utilization cap | Sensitivity to data choices (F2, F6) | Pending |
| VIF, Box-Tidwell | Assumption tests | Pending |

**What was possible without labels** (cs-test): the code-execution test and data profile above, and a **score-distribution check** which can only be done with the real artifacts. For ongoing use, the same file is suitable as a population-stability reference.

#### *vii. Benchmark / Challenger Model*

**Status: pending.** Planned: gradient-boosted trees (`HistGradientBoostingClassifier`) with class weighting, plus a logistic regression with sentinel handling, income-missing indicator and log-transformed ratios, evaluated on identical splits for AUC, KS and calibration.

| Metric | Developer model | Challenger model |
|---|---|---|
| AUC (hold-out) | 0.8088 (developer-reported) | Pending |
| KS | Not reported | Pending |
| Calibration slope | Not reported | Pending |

*Expectation (not a result):* public results on this dataset typically show tree ensembles ahead of plain logistic regression by several AUC points, mainly from the skewed variables and missing-income structure; the developer's claim that alternatives were unsuitable should be tested, not assumed.

#### *viii. Limitations Review*
*Mirrors developer section vi (limitations)*

Stated: single schema / exact column names; low precision and recall. **Not disclosed:**
- The training/scoring capping mismatch (F2) and the in-sample metrics (F3).
- Sentinel codes 96/98 treated as counts; `DebtRatio` is not a ratio when income is missing (F6).
- Median imputation for ~20% missing income, with no missingness indicator.
- Unknown data source and period; no evidence of fit to the bank's portfolio (F9).
- No calibration evidence; unweighted rare-event fit; no challenger (F7).
- Two-year serious delinquency is not a regulatory default definition.
- Output table's "Borrower" is a row counter, not an identifier.
- No fairness / protected-class review; `age` is used directly as a predictor, which may be restricted in some credit-decision contexts (needs policy/legal confirmation).

#### *ix. Ongoing Monitoring Review*
*Mirrors developer section vi (monitoring)*

The monitoring section says only that data should be "kept up to date" and that columns may need cleaning. There are **no metrics and no trigger levels** (F8). Minimum expected set: rolling AUC/Gini and KS on matured cohorts (trigger e.g. AUC fall of >0.05 vs baseline); calibration (predicted vs observed by decile, with a binomial test); PSI per feature and on the score (amber 0.10, red 0.25); missing-value and out-of-range rates (e.g. share of 96/98 codes, missing income); and a re-validation schedule.

#### *x. Findings*

| Rating | Meaning | Example |
|---|---|---|
| High | Results cannot be trusted or reproduced | Code does not run; data leakage |
| Medium | Usable, but a real weakness | Untested assumption; documentation does not match code |
| Low | Documentation or presentation issue | Wrong date; unclear wording |

| ID | Section | Finding | Evidence (cell, number or test) | Rating | Required action |
|---|---|---|---|---|---|
| F1 | v. Replication | Delivered `model.py` fails: `median_depend` and `upper_bound` are never defined or saved, and the pickles are not in the repo | Run in section v: `NameError: name 'median_depend' is not defined`; notebook cell 1 saves only 3 of 5 needed objects | High | Save all five artifacts, fix `model.py` (corrected version provided), add a test that scores `cs-test.csv` end to end |
| F2 | iii / iv | Training and scoring use different inputs. Utilization/income clipping is applied to `training_data` *after* `x` was built, so the model is fitted on unclipped data, yet `predict_pd` clips utilization at the 85th percentile (income clip is not applied at all). The bound is also computed on the full data, including hold-out rows | `model.ipynb` cell 1 (order of `x = ...` vs `clip`), cell 3 and `model.py` line 29 | High | Decide the treatment, apply it identically in training and scoring (bounds from training split only), justify the percentiles, refit and re-report all metrics |
| F3 | v. Replication | Reported recall, precision and accuracy come from scoring the full training file (about 80% used for fitting) after it was mutated by the clip; they are not hold-out results. The documentation also names `cs-test.csv` as the validation set although it has no target labels | `model.ipynb` cell 5 (`predict_pd(training_data)`); cs-test: 0 non-null targets (section iii) | High | Recompute on `x_test`/`y_test`, report AUC, KS, Gini and calibration; correct the validation-data statement |
| F4 | v / vi | Decision threshold is inconsistent (20% in code, 25% in text, 50% in a comment) and not tied to a business cost | `model.ipynb` cell 5; documentation "Pre-Validation Testing" | Medium | Choose one cut-off using a cost-based or capacity-based rule on a validation set; show sensitivity |
| F5 | ii / iv | Documentation does not describe what the code does: dependents imputation, both clipping steps, and the percentiles are undocumented; the no-multicollinearity assumption is stated but never tested; "none of the other models produce probabilities" is incorrect | Documentation sections "Data Description" and "Logistic Regression Model Details" vs `model.ipynb` cell 1 | Medium | Update the documentation to match the final code; add the assumption tests |
| F6 | iii / iv | Data-quality defects unhandled: 214 rows with 96/98 sentinel codes inflate the three delinquency counts and drive their 0.99 correlation; `DebtRatio` holds dollar amounts for rows with missing income (18,849 of 20,103 such rows are > 1), so median income imputation does not repair it | Section iii output; section iv correlation table | Medium | Recode sentinels as missing/flag, add a missing-income indicator, treat `DebtRatio` for missing income separately, report VIF |
| F7 | iv / vii | Model choices unjustified: no challenger, invalid reason for rejecting alternatives, no handling of ~7% event rate, no calibration test, default hyper-parameters, no log/winsorising of heavy-tailed predictors | Documentation section iv; `model.ipynb` cell 1 | Medium | Build and compare a challenger; test calibration and linearity; document tuning |
| F8 | ix. Monitoring | No monitoring metrics or trigger levels | Documentation "Ongoing monitoring" paragraph | Medium | Define metrics, thresholds and owners (see section ix) |
| F9 | ii. Intended use | Training population (2011 Kaggle data, unknown source and period) is not shown to represent the bank's existing borrowers; use cases and prohibited uses are not stated; `age` used as a predictor without policy review | Documentation "Data Description" and "Scope" | Medium | Document and justify representativeness, state permitted/prohibited uses, obtain compliance sign-off on `age` |
| F10 | v. Replication | Not reproducible by a third party: absolute laptop paths, no `requirements.txt`/library versions, pickles and training data not in the repo, same function duplicated in notebook and `model.py` (and the copies drifted, F1) | `model.ipynb` cells 0, 3, 5 | Low | Relative paths, pinned environment, commit artifacts or a training script, single source for `predict_pd` |
| F11 | x. Presentation | Wording and consistency: "this model kind of sucks", "6%" vs ~6.7% prevalence, hard-coded "AUC = 0.81" legend, code comment says "high recall, low precision" while the text says both are low, accuracy quoted without the 93% naive baseline | Documentation "Limitations"; `model.ipynb` cells 5–7 | Low | Edit for accuracy and professional tone; report accuracy against the naive baseline |

#### *xi. Overall Conclusion*

**Approved with conditions.** The logistic regression baseline is an appropriate choice and the documentation is clear about purpose and data. The model should not be put into use until the following conditions are met:

1. **F1:** `model.py` runs end to end (a corrected version is provided; the developer should save and commit the missing artifacts).
2. **F2:** the capping treatment is applied identically in training and scoring, and the model is refit.
3. **F3:** performance metrics are recomputed on the hold-out set, and the validation-data statement is corrected.

After these, the validator will complete replication, AUC/KS/calibration, stability and challenger testing (sections v–vii) using `cs-training.csv`. Medium findings F4–F9 should be closed or accepted with conditions within the agreed timeline; Low findings F10–F11 are housekeeping.

#### *xii. Developer Response*
*Completed by the model developer*

| ID | Agree / Dispute | Response or action taken | Date |
|---|---|---|---|
| F1 | | | |
| F2 | | | |
| F3 | | | |
| F4 | | | |
| F5 | | | |
| F6 | | | |
| F7 | | | |
| F8 | | | |
| F9 | | | |
| F10 | | | |
| F11 | | | |

**Validator sign-off:** all findings F1–F11 are **open** pending developer response; approval is conditional on closure of F1–F3.